# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and display metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In a Croissant dataset, the main data is organized into *record sets* (tables). Each record set and its fields can be uniquely referenced via their `@id`. We'll enumerate the record sets, display their `@id`s and list their fields.

In [ ]:
# List all record sets and their `@id`s
record_sets = dataset.record_sets
print("Record sets:")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
    # List fields for this record set
    if 'field' in rs:
        print("  Fields:")
        for fld in rs['field']:
            print(f"    * {fld['@id']} (name: {fld.get('name', 'N/A')})")
    else:
        print("  No fields available.")

# For illustration, display a few records from the first record set
if len(record_sets) > 0:
    example_rs_id = record_sets[0]['@id']
    print(f"\nSample records from record set {example_rs_id}:")
    for record in dataset.records(record_set=example_rs_id):
        print(record)
        break  # display only the first record

## 3. Data Extraction
Load data from record sets into DataFrames for analysis.
We use `@id`s of record sets and fields for accessing and extracting the data.

Below, we demonstrate loading all available record sets (usually just one main table for survey datasets) into pandas DataFrames.

In [ ]:
# Prepare a list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for {rs_id}:")
    print(df.columns.tolist())
    print(df.head())

# For demonstration, select the first record set
primary_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[primary_record_set_id] if primary_record_set_id else pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Analyze and process the data.

We'll filter records by a numeric field, normalize it, and group by a categorical field.
All fields are referenced by their `@id`. Below, inspect the available columns and select example field IDs for numeric and grouping operations.

In [ ]:
# Inspect column names to select field IDs for EDA
print("Available columns:", df.columns.tolist())

# Example field IDs -- update as needed based on your dataset's actual columns
# For illustration, suppose GAD-7 scores are under '@id': 'gad_7_total_score', grouping by 'gender'
numeric_field_id = 'gad_7_total_score'  # replace with true field @id
group_field_id = 'gender'               # replace with true field @id

# If these fields exist, perform EDA
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field_id}' not found. Please update numeric_field_id and group_field_id.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

We'll plot histogram and boxplot visualizations for a numeric field, grouped by a categorical field, using their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot grouped by a categorical field
    if group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print(f"Cannot visualize. Field '{numeric_field_id}' not found in DataFrame.")

## 6. Conclusion
This notebook provided an overview, extraction, and preliminary analysis of the FAIR² Mental Health Survey Dataset from Kilifi County using the `mlcroissant` library.

- Data was loaded and referenced by Croissant `@id`.
- We demonstrated filtering and normalization of numeric fields, and aggregation by a group field.
- Initial visualizations highlighted distributions and group differences.

Further analyses can be performed by referencing dataset schema and `@id`s and extending the EDA for your research.